<!-- NB_DOC_INTRO_V1 -->
# AI Engineering — Fine-tuning LLMs (PEFT / LoRA / QLoRA / SFT / DPO)

> 📚 **Doc thematique** : [docs/09_AI_ENGINEERING.md](docs/09_AI_ENGINEERING.md) (AI Engineering)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Fine-tuner un LLM** sans GPU haut de gamme = inaccessible historiquement. **PEFT** (Parameter-Efficient Fine-Tuning) change la donne : avec **LoRA** (16 GB VRAM) ou **QLoRA** (6 GB VRAM, base model en 4-bit), on adapte un Llama-3 8B sur un GPU consumer (RTX 4060+) ou un Mac M2 Pro.

Ce notebook couvre l'**ecosysteme 2024-2025** (HuggingFace transformers + peft + trl + bitsandbytes + accelerate) avec code execute pour les parties metadata + dataset prep (CPU OK), et code de reference complet pour le training (necessite GPU). Couvre : SFT (Supervised Fine-Tuning), **DPO** (Direct Preference Optimization, alignement sans reward model), et evaluation.

**Conseils strategiques** : essayer **Prompt Engineering** → **Few-shot** → **RAG** AVANT le fine-tuning. FT = quand on a besoin de **style/format/vocabulaire persistant**.

Versions : `transformers >= 4.40`, `peft >= 0.9`, `trl >= 0.7`.

## Plan

1. Echelle des approches (prompt eng → FT → from-scratch)
2. Methodes PEFT (LoRA, QLoRA, prefix tuning, adapter, IA3)
3. Stack moderne (HuggingFace transformers + peft + trl + bitsandbytes)
4. **Demo** : preparation dataset au format chat (executable CPU)
5. Configuration QLoRA (code de reference GPU)
6. SFT (Supervised Fine-Tuning) — code complet
7. DPO (Direct Preference Optimization) — alignement
8. Inference avec adapter LoRA
9. Quantization post-FT (GGUF, AWQ) pour deploiement
10. Datasets publics utiles
11. Evaluation (MT-Bench, lm-evaluation-harness)
12. Couts indicatifs cloud
13. Pieges et anti-patterns
14. References


## 1. Echelle des approches de specialisation

| # | Approche | Cout | Quand utiliser |
|---|---|---|---|
| 1 | **Prompt engineering** | ~ 0 | Le modele sait deja, il faut juste le guider |
| 2 | **Few-shot prompting** | ~ 0 | Quelques exemples dans le prompt |
| 3 | **RAG** | Bas | Ancrer dans des sources specifiques (facts) |
| 4 | **Fine-tuning (PEFT)** | Moyen | Style / format / vocabulaire persistant |
| 5 | **Continued pre-training** | Eleve | Nouveau domaine (medical, juridique) avec corpus massif |
| 6 | **From-scratch training** | Enorme ($M) | Reserve aux labos / GAFAM |

> 🎯 **Regle** : essayer PE → FS → RAG AVANT FT. FT n'est PAS un substitut a RAG (facts) — c'est complementaire.

## 2. Methodes PEFT — comparatif

| Methode | Idee | Memoire | Qualite vs full FT |
|---|---|---|---|
| **Full FT** | Reentrainer **tous** les poids | ~ 100 GB VRAM pour 7B | 100% |
| **LoRA** (Hu et al. 2021) | Adapter low-rank : `W' = W + AB` (A, B petits) | ~ 16 GB VRAM (7B FP16) | ~ 95% |
| **QLoRA** (Dettmers 2023) | LoRA + **base model en 4-bit NF4** | ~ 6 GB VRAM (7B) | ~ 95% |
| **Prefix Tuning** | Prepend des tokens "virtuels" appris | Tres leger | ~ 90% |
| **Prompt Tuning** | Idem mais 1 seule couche | Leger | ~ 85% |
| **Adapter Layers** (Houlsby 2019) | Petits modules entre couches | Leger | ~ 92% |
| **IA3** | Multiplier les activations par vecteurs appris | Ultra-leger | ~ 92% |

> 🔥 **2024-2025** : QLoRA est le standard de facto pour fine-tuning sur GPU consumer / Mac.

## 3. Stack moderne

| Lib | Role |
|---|---|
| **`transformers`** (HuggingFace) | Modeles + tokenizers + Trainer |
| **`peft`** | LoRA / QLoRA / IA3 / Prefix tuning |
| **`bitsandbytes`** | Quantization 4/8-bit (CUDA Linux only) |
| **`trl`** | SFT (Supervised Fine-Tuning), **DPO**, PPO |
| **`accelerate`** | Multi-GPU, mixed precision, distributed |
| **`datasets`** | Gestion des datasets HF |
| **`unsloth`** (alternatif) | 2-5× plus rapide, GPU consumer |


## 4. Demo executable — preparation dataset au format chat

C'est la **partie CPU** : on prepare un dataset au bon format avant FT.


In [ ]:
# pip install -q transformers
from transformers import AutoTokenizer
import json

# === Charger un tokenizer (sans le modele complet, leger) ===
# Mistral-7B-Instruct utilise le format [INST] question [/INST] reponse
TOKENIZER_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

try:
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
    print(f"Tokenizer charge : {TOKENIZER_NAME}")
    print(f"Vocab size : {tokenizer.vocab_size}")
    print(f"Special tokens : {tokenizer.special_tokens_map}")
except Exception as e:
    print(f"Tokenizer load failed (besoin internet HF) : {type(e).__name__}")
    print("Continuer avec un fallback simple...")
    tokenizer = None

In [ ]:
# === Dataset jouet : 5 exemples Q/R ===
raw_dataset = [
    {"instruction": "Explique le RAG en une phrase.",
     "output": "Le RAG combine un retriever et un LLM pour ancrer les reponses dans des sources."},
    {"instruction": "Quelle est la difference entre LoRA et QLoRA ?",
     "output": "LoRA entraine des adapters low-rank, QLoRA quantize en plus le base model en 4-bit."},
    {"instruction": "Liste 3 vector stores populaires.",
     "output": "FAISS, Chroma, Qdrant."},
    {"instruction": "Pourquoi utiliser un cross-encoder pour le re-ranking ?",
     "output": "Il encode (query, doc) ensemble donc plus precis qu'un bi-encoder, mais coute O(N) au runtime."},
    {"instruction": "Donne le format de chat pour Llama 3.",
     "output": "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nQUESTION<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nREPONSE<|eot_id|>"},
]

# === Formater pour Mistral instruct ===
def format_mistral(example: dict) -> str:
    return f"[INST] {example['instruction']} [/INST] {example['output']}{'</s>' if tokenizer else ''}"

formatted = [format_mistral(ex) for ex in raw_dataset]
print(f"=== Exemple formate Mistral ===\n{formatted[0]}\n")

# === Compter les tokens (si tokenizer dispo) ===
if tokenizer:
    lengths = [len(tokenizer.encode(t)) for t in formatted]
    print(f"Token lengths : {lengths}")
    print(f"Total tokens dans le dataset : {sum(lengths)}")

### 4.1 Format **chat template** (recommande, agnostique)

Au lieu d'ecrire les balises manuellement, utiliser le **chat template** du modele.


In [ ]:
if tokenizer:
    # Format chat avec le template Hugging Face (correct pour chaque modele)
    messages = [
        {"role": "user", "content": "Explique le RAG en une phrase."},
        {"role": "assistant", "content": "Le RAG combine un retriever et un LLM..."},
    ]
    formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)
    print(f"Apres apply_chat_template :\n{formatted_text}")
else:
    print("Tokenizer non charge, skip.")

## 5. Configuration QLoRA (code GPU)

```python
# pip install -q transformers peft bitsandbytes trl accelerate datasets
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset

# === 1. Quantization config (QLoRA = 4-bit NF4) ===
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",                # NF4 = NormalFloat 4-bit
    bnb_4bit_compute_dtype=torch.bfloat16,    # BF16 pour calculs
    bnb_4bit_use_double_quant=True,           # double quant = plus compact
)

# === 2. Charger modele en 4-bit ===
model_name = "mistralai/Mistral-7B-Instruct-v0.2"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb,
    device_map="auto",                        # GPU auto
)
model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
```

### Configurer LoRA
```python
lora = LoraConfig(
    r=16,                                     # rang — 8/16/32 selon RAM (compromis qualite/cout)
    lora_alpha=32,                            # scaling (typiquement 2 × r)
    target_modules=["q_proj", "v_proj"],      # quelles couches "LoRA-ifier" ?
                                               # Mistral : ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()
# trainable params: 4,194,304 || all params: 7,245,732,352 || trainable%: 0.0579
# ~ 0.06% des params, mais souvent ~ 95% de la qualite du full FT
```


## 6. SFT (Supervised Fine-Tuning)

```python
# === Dataset au format chat ===
def format_alpaca(example):
    if example.get("input"):
        prompt = f"[INST] {example['instruction']}\n\nContexte: {example['input']} [/INST]"
    else:
        prompt = f"[INST] {example['instruction']} [/INST]"
    text = f"{prompt} {example['output']}{tokenizer.eos_token}"
    return {"text": text}

ds = load_dataset("yahma/alpaca-cleaned", split="train")
ds = ds.map(format_alpaca, remove_columns=ds.column_names)
# ds = ds.select(range(2000))   # subset pour rapide

# === Training arguments ===
args = TrainingArguments(
    output_dir="./mistral-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,           # batch effectif 16
    learning_rate=2e-4,
    bf16=True,                                # ou fp16=True (selon GPU)
    optim="paged_adamw_8bit",                # optimizer 8-bit, gain RAM
    logging_steps=10,
    save_strategy="epoch",
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="wandb",                       # log dans W&B
    gradient_checkpointing=True,             # gain RAM x2-3, temps +20%
)

# === Trainer ===
trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    args=args,
    tokenizer=tokenizer,
    max_seq_length=2048,
    dataset_text_field="text",
)

trainer.train()
trainer.model.save_pretrained("./mistral-lora-final")
```


## 7. DPO (Direct Preference Optimization)

Pour **aligner** le modele (helpful, harmless, honest) :
- Fournir des paires `(chosen, rejected)` (= reponse preferee / rejetee)
- DPO entraine le modele a preferer la bonne directement, **sans reward model** (contrairement a RLHF/PPO)

```python
from trl import DPOTrainer

# Dataset : {"prompt": ..., "chosen": ..., "rejected": ...}
# Ex : HuggingFaceH4/ultrafeedback_binarized
dpo_dataset = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs")

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,             # PEFT : utilise base model comme ref (gain RAM)
    beta=0.1,                    # poids du KL divergence vs ref model
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
    args=args,
)
dpo_trainer.train()
```

### Strategie complete recommandee
```
1. SFT sur instruction dataset    → modele suit les instructions
2. DPO sur preferences            → modele aligne (helpful, harmless, honest)
3. Eval sur benchmarks (MMLU, MT-Bench, custom)
4. Quantize (GGUF / AWQ / GPTQ)   → deploiement leger
```


## 8. Inference avec le modele fine-tuned

```python
from peft import PeftModel

# Charger base + adapter
base = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb, device_map="auto")
model = PeftModel.from_pretrained(base, "./mistral-lora-final")

# Inference
prompt = "[INST] Explique le LoRA en une phrase. [/INST]"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0], skip_special_tokens=True))

# Merger LoRA dans le base (pour deploiement standalone)
merged = model.merge_and_unload()
merged.save_pretrained("./mistral-merged")
```


## 9. Quantization post-FT pour deploiement

Apres merge, on quantize en **GGUF Q4_K_M** pour Ollama / llama.cpp, ou **AWQ** pour vLLM.

```bash
# GGUF
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp
python convert.py path/to/merged
quantize merged.gguf merged-q4_k_m.gguf Q4_K_M
ollama create my-custom -f Modelfile   # avec "FROM merged-q4_k_m.gguf"

# AWQ pour vLLM
pip install autoawq
# script python conv standard
```


## 10. Datasets publics utiles

| Dataset | Type | Taille | Langue |
|---|---|---:|---|
| `yahma/alpaca-cleaned` | Instruction | 52k | EN |
| `OpenAssistant/oasst1` | Conversations multi-tours | 91k | Multi |
| `databricks/databricks-dolly-15k` | Instructions humaines | 15k | EN |
| `HuggingFaceH4/ultrachat_200k` | Chat synthetique | 200k | EN |
| `Anthropic/hh-rlhf` | Preferences (helpful/harmful) | 161k pairs | EN |
| `croissantllm/croissant_dataset` | Multilingue (FR) | M | FR/EN |
| `tatsu-lab/alpaca` | Instructions original | 52k | EN |
| `meta-math/MetaMathQA` | Math reasoning | 395k | EN |
| `argilla/distilabel-intel-orca-dpo-pairs` | DPO pairs | 12k | EN |

## 11. Evaluation post-FT

- **MT-Bench** (LLM-as-judge) — 80 questions multi-turn evaluees par GPT-4
- **lm-evaluation-harness** (EleutherAI) — MMLU, HellaSwag, ARC, TruthfulQA, GSM8K
- **Custom dataset eval** : tes propres examples gold
- **AlpacaEval** : automatique vs GPT-4

```bash
# MT-Bench
pip install fastchat
python -m fastchat.llm_judge.gen_model_answer --model-path ./mistral-merged --model-id custom
python -m fastchat.llm_judge.gen_judgment --model-list custom --judge gpt-4
python -m fastchat.llm_judge.show_result --model-list custom
```

## 12. Couts indicatifs cloud (juin 2024)

| Setup | Temps | Cout (cloud) |
|---|---|---|
| QLoRA Mistral 7B, 1 epoch sur 5k samples, A100 40GB | ~ 1h | ~ $1.5 (Lambda/Vast) |
| LoRA Llama-3 70B sur A100×4 | ~ 6h | ~ $20-40 |
| Full FT Mistral 7B, 8×A100 | ~ 12h | ~ $100+ |
| Mac M2 Pro 32 GB (QLoRA 7B avec MLX-LM) | ~ 4h | $0 (electricite) |


## 13. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correction |
|---|---|
| FT pour donner des connaissances factuelles | Utiliser **RAG** plutot |
| Trop d'epochs | Overfitting et "catastrophic forgetting" — typiquement 1-3 epochs |
| LR trop eleve | Modele "casse" (perte des capacites generales) — typiquement 1e-4 a 5e-5 |
| Dataset bruite | Modele apprend les erreurs — clean d'abord |
| Pas de eval avant/apres | Tu ne sais pas si ca marche |
| LoRA r=128 sur petit dataset | Overfit. r=8-16 suffit generalement |
| `target_modules=["q_proj"]` seul | Manque k_proj/v_proj/o_proj. Pour Mistral : tous |
| Pas de chat template lors de l'inference | Outputs cates au format brut |
| Oublier `tokenizer.pad_token = tokenizer.eos_token` | Erreur de batching |
| bf16 sur GPU pre-Ampere | Pas supporte, utiliser fp16 |
| Pas de gradient_checkpointing si VRAM serree | OOM |
| `accumulate_grad_batches` mal utilise | Toujours batch effectif >= 16 |

## 14. References

### Documentation
- **HuggingFace PEFT** : https://huggingface.co/docs/peft
- **TRL** : https://huggingface.co/docs/trl
- **transformers** : https://huggingface.co/docs/transformers
- **Unsloth** (alternative rapide) : https://github.com/unslothai/unsloth
- **Axolotl** (high-level wrapper) : https://github.com/OpenAccess-AI-Collective/axolotl

### Papers fondateurs
- Hu et al. (2021). *LoRA: Low-Rank Adaptation of Large Language Models*.
- Dettmers et al. (2023). *QLoRA: Efficient Finetuning of Quantized LLMs*.
- Rafailov et al. (2023). *Direct Preference Optimization*.
- Houlsby et al. (2019). *Parameter-Efficient Transfer Learning for NLP* (Adapter).
- Liu et al. (2022). *Few-Shot Parameter-Efficient Fine-Tuning is Better and Cheaper than In-Context Learning* (IA3).

### Tutos pratiques
- HuggingFace community : https://huggingface.co/learn
- Maxime Labonne — LLM Course : https://github.com/mlabonne/llm-course

Voir aussi :
- [NLP_Transformers.ipynb](./NLP_Transformers.ipynb) — bases Transformers
- [AI_Local_LLMs.ipynb](./AI_Local_LLMs.ipynb) — deployer ton modele FT
- [NLP_LangChain_RAG.ipynb](./NLP_LangChain_RAG.ipynb) — RAG (alternative au FT)
- [AI_Prompt_Engineering.ipynb](./AI_Prompt_Engineering.ipynb) — essayer avant FT


<!-- ENRICH_2025_V1 -->

## 🔥 Outils de fine-tuning 2024-2025

### 1. **Unsloth** — 2-5× plus rapide que HF transformers/PEFT

[Unsloth](https://github.com/unslothai/unsloth) reecrit les kernels CUDA pour Llama / Mistral / Phi / Gemma. **Gain enorme** sur GPU consumer.

```python
# pip install unsloth
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16, lora_dropout=0, use_gradient_checkpointing="unsloth",
)
# Puis HF Trainer standard, 2-5× plus rapide.
```

Sur **un Mac M2/M3** : MLX-LM (Apple) joue le meme role natif.

### 2. **ORPO** (Odds Ratio Preference Optimization, juin 2024)

Une **seule etape** SFT + alignment (vs SFT puis DPO). Plus simple, souvent meilleur sur petits datasets.

```python
from trl import ORPOConfig, ORPOTrainer
config = ORPOConfig(output_dir="./out", beta=0.1, ...)
trainer = ORPOTrainer(model=model, args=config, ...)
```

### 3. **KTO** (Kahneman-Tversky Optimization)

Comme DPO mais accepte des **labels binaires** ("bon" / "mauvais") au lieu de paires (chosen, rejected). Plus facile a labelliser.

### 4. **SimPO** (octobre 2024)

DPO sans modele de reference (encore plus leger). Souvent equivalent ou meilleur que DPO.

### 5. **Axolotl** — framework YAML pour FT

Au lieu de coder le training loop, on declare en YAML :
```yaml
base_model: meta-llama/Llama-3.1-8B
load_in_4bit: true
adapter: qlora
lora_r: 64
sequence_len: 2048
datasets:
  - path: tatsu-lab/alpaca
    type: alpaca
optimizer: paged_adamw_8bit
lr_scheduler: cosine
```
Puis `accelerate launch -m axolotl.cli.train config.yaml`. Standard dans la communaute open-source.

### 6. **LoRA-Pro / DoRA / GaLore** (Variants 2024)

- **DoRA** : decompose les weights en magnitude + direction → souvent meilleur que LoRA pure
- **GaLore** : memoire encore reduite (gradient low-rank projection)
- **LongLoRA** : pour fine-tuning long context

## 🔥 Datasets pour FT en 2025

| Dataset | Type | Taille | Langue |
|---|---|---:|---|
| **OpenHermes 2.5** | Chat haute qualite | 1M | EN |
| **UltraChat** | Multi-turn | 200k | EN |
| **Distilabel** | Synthetic, DPO pairs | varies | EN |
| **Capybara** | Long context | 16k | EN |
| **Tatsu Alpaca / Dolly** | Classique instruction | 52k / 15k | EN |
| **CroissantLLM** | Multi (FR) | M | FR/EN |

## 💡 Recap quoi/quand

| Cas | Approche |
|---|---|
| **POC rapide** | QLoRA + Unsloth (1-2h sur GPU consumer) |
| **Alignment qualite** | SFT puis DPO (ou ORPO en 1 step) |
| **GPU 24 GB consumer** | QLoRA 7-13B parfait |
| **Mac M-series** | MLX-LM (Apple native) |
| **Cluster GPU** | Axolotl YAML + multi-GPU |
| **Datasets de preferences** | OpenAssistant, UltraFeedback, distilabel |
